# Gini — Ghost Aviator 3D Model Generator

Turns the Ghost Aviator hero artwork into a real, textured 3D mesh (`.glb`)
using a GPU, then hands it back for rigging and animation in Blender.

**Before you run anything:** Runtime → Change runtime type → **T4 GPU** → Save.

Then just run the cells top to bottom. Cell 3 will ask you to upload
`char_1024.png` (it is in `ghost-aviator/tools/gini/input/`).

In [ ]:
#@title 1. Check the GPU  { display-mode: "form" }
# If this errors or shows no GPU, set Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU'

In [ ]:
#@title 2. Install Hunyuan3D-2 (takes ~5-8 minutes)  { display-mode: "form" }
%cd /content
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git 2>/dev/null || echo 'already cloned'
%cd /content/Hunyuan3D-2

# Core deps. Colab already ships torch + CUDA, so we do NOT reinstall torch
# (that is the classic way to break a Colab runtime).
!pip install -q ninja pybind11 trimesh pymeshlab pygltflib xatlas
!pip install -q diffusers transformers accelerate safetensors einops omegaconf
!pip install -q opencv-python rembg onnxruntime
print('deps installed')

In [ ]:
#@title 2b. Finish the install (REQUIRED)  { display-mode: "form" }
# `pip install -e .` registers the repo's own package. Without it,
# `import hy3dgen` fails in cell 5. The two setup.py builds add texture
# support and compile CUDA extensions -- they are allowed to fail, because
# an untextured mesh is still perfectly usable and can be shaded in Blender.
import subprocess, sys, os

%cd /content/Hunyuan3D-2
!pip install -q -e .

# The texture extensions are OPTIONAL and their paths vary between revisions
# of this repo -- the README documents hy3dgen/texgen/custom_rasterizer, which
# does not exist in every clone. Skip anything missing instead of raising:
# an untextured mesh is still exactly what we need, and Blender can shade it.
for sub in ['hy3dgen/texgen/custom_rasterizer',
            'hy3dgen/texgen/differentiable_renderer']:
    path = f'/content/Hunyuan3D-2/{sub}'
    if not os.path.isdir(path):
        print(f'skip {sub} (not present in this clone)')
        continue
    print('building', sub, '...')
    r = subprocess.run([sys.executable, 'setup.py', 'install'],
                       cwd=path, capture_output=True, text=True)
    print('   OK' if r.returncode == 0 else
          f'   FAILED (texture will be skipped)\n{r.stderr[-500:]}')

# THE check that decides whether cell 5 can run.
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
print('\nSHAPE PIPELINE IMPORT OK -- cell 5 will work')


In [ ]:
#@title 3. Upload the character image  { display-mode: "form" }
# Upload: ghost-aviator/tools/gini/input/char_1024.png
from google.colab import files
import os, shutil

os.makedirs('/content/gini_in', exist_ok=True)
up = files.upload()
src = list(up.keys())[0]
IMG = '/content/gini_in/char.png'
shutil.move(src, IMG)

from PIL import Image
im = Image.open(IMG)
print('uploaded:', IMG, im.size, im.mode)
im.thumbnail((320, 320))
display(im)

In [ ]:
#@title 4. Remove the background (isolate the character)  { display-mode: "form" }
# The hero art is a full scene (mountains, sunset). Without this the generator
# happily reconstructs the mountains too.
from rembg import remove
from PIL import Image

IMG = '/content/gini_in/char.png'
CUT = '/content/gini_in/char_cut.png'

src = Image.open(IMG).convert('RGBA')
out = remove(src)
out.save(CUT)
print('saved', CUT, out.size)

prev = out.copy(); prev.thumbnail((320, 320))
display(prev)
print('CHECK THIS: the character should be isolated with nothing of the mountains left.')

In [ ]:
#@title 5. Generate the 3D shape  { display-mode: "form" }
octree_resolution = 320 #@param {type:"slider", min:192, max:512, step:32}
inference_steps   = 50  #@param {type:"slider", min:20, max:100, step:5}

%cd /content/Hunyuan3D-2
import torch, time
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')

t0 = time.time()
mesh = pipe(
    image='/content/gini_in/char_cut.png',
    num_inference_steps=inference_steps,
    octree_resolution=octree_resolution,
    guidance_scale=5.0,
    generator=torch.manual_seed(1234),
)[0]
print(f'shape generated in {time.time()-t0:.0f}s')

mesh.export('/content/gini_shape.glb')
print('saved /content/gini_shape.glb')

In [ ]:
#@title 6. Paint the texture (optional but worth it)  { display-mode: "form" }
# If this cell runs out of VRAM on a T4, skip it — the untextured shape from
# cell 5 is still usable, and Blender can shade it.
try:
    from hy3dgen.texgen import Hunyuan3DPaintPipeline
    paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
    textured = paint(mesh, image='/content/gini_in/char_cut.png')
    textured.export('/content/gini_textured.glb')
    print('saved /content/gini_textured.glb')
except Exception as e:
    print('texture step failed (this is survivable):', type(e).__name__, e)

In [ ]:
#@title 7. Download the model  { display-mode: "form" }
from google.colab import files
import os

for p in ['/content/gini_textured.glb', '/content/gini_shape.glb']:
    if os.path.exists(p):
        print(f'{p}  {os.path.getsize(p)/1e6:.2f} MB')
        files.download(p)

print('\nSave the .glb into: ghost-aviator/tools/gini/out/')